In [1]:
# Run this in the first cell of your notebook
!pip install s3fs boto3 delta-spark==3.1.0

INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 902.5 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 2.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 11.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from pyspark.sql import SparkSession

jar_paths = [
    "/home/jovyan/work/jars/delta-spark_2.12-3.1.0.jar",
    "/home/jovyan/work/jars/delta-storage-3.1.0.jar",
    "/home/jovyan/work/jars/hadoop-aws-3.3.4.jar",
    "/home/jovyan/work/jars/aws-java-sdk-bundle-1.12.262.jar"
]
jars_string = ",".join(jar_paths)

spark = SparkSession.builder \
    .appName("LakehouseSetup_Offline") \
    .config("spark.jars", jars_string) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print("Connected successfully in Offline Mode! Ready to build the Lakehouse.")

Spark Version: 3.5.0
Connected successfully in Offline Mode! Ready to build the Lakehouse.


In [3]:
# import os
# import uuid
# import random
# import pandas as pd
# import numpy as np
# from datetime import datetime, timedelta

# # 1. Setup & Configuration
# LANDING_DIR = "./landing"
# os.makedirs(LANDING_DIR, exist_ok=True)

# NUM_CUSTOMERS = 50
# NUM_SELLERS = 10
# NUM_PRODUCTS = 20
# NUM_ORDERS = 100
# SHARED_ZIP_CODES = [random.randint(10000, 99999) for _ in range(20)]
# def generate_id():
#     return uuid.uuid4().hex

# def random_date(start_date, end_date):
#     time_between_dates = end_date - start_date
#     days_between_dates = time_between_dates.days
#     random_number_of_days = random.randrange(days_between_dates)
#     return start_date + timedelta(days=random_number_of_days)

# # 2. Generate Dimension Tables (Primary Keys)
# print("Generating Dimension Tables...")

# # Customers & CRM Identities
# customer_ids = [generate_id() for _ in range(NUM_CUSTOMERS)]
# customer_unique_ids = [generate_id() for _ in range(NUM_CUSTOMERS)]
# states = ['SP', 'RJ', 'MG', 'RS', 'PR']

# customers_df = pd.DataFrame({
#     'customer_id': customer_ids,
#     'customer_unique_id': customer_unique_ids,
#     'customer_zip_code_prefix': [random.choice(SHARED_ZIP_CODES) for _ in range(NUM_CUSTOMERS)],
#     'customer_city': [random.choice(['sao paulo', 'rio de janeiro', 'belo horizonte', 'curitiba']) for _ in range(NUM_CUSTOMERS)],
#     'customer_state': [random.choice(states) for _ in range(NUM_CUSTOMERS)]
# })

# crm_df = pd.DataFrame({
#     'customer_unique_id': customer_unique_ids,
#     'email': [f"user_{str(uuid.uuid4())[:8]}@example.com" for _ in range(NUM_CUSTOMERS)]
# })

# # Sellers
# seller_ids = [generate_id() for _ in range(NUM_SELLERS)]
# sellers_df = pd.DataFrame({
#     'seller_id': seller_ids,
#     'seller_zip_code_prefix': [random.choice(SHARED_ZIP_CODES) for _ in range(NUM_SELLERS)],
#     'seller_city': [random.choice(['sao paulo', 'campinas', 'florianopolis']) for _ in range(NUM_SELLERS)],
#     'seller_state': [random.choice(states) for _ in range(NUM_SELLERS)]
# })

# # Products
# product_ids = [generate_id() for _ in range(NUM_PRODUCTS)]
# categories = ['cama_mesa_banho', 'esporte_lazer', 'moveis_decoracao', 'beleza_saude', 'informatica_acessorios']
# products_df = pd.DataFrame({
#     'product_id': product_ids,
#     'product_category_name': [random.choice(categories) for _ in range(NUM_PRODUCTS)],
#     'product_name_lenght': np.random.randint(20, 60, NUM_PRODUCTS),
#     'product_description_lenght': np.random.randint(100, 1500, NUM_PRODUCTS),
#     'product_photos_qty': np.random.randint(1, 5, NUM_PRODUCTS),
#     'product_weight_g': np.random.randint(100, 5000, NUM_PRODUCTS),
#     'product_length_cm': np.random.randint(15, 50, NUM_PRODUCTS),
#     'product_height_cm': np.random.randint(5, 30, NUM_PRODUCTS),
#     'product_width_cm': np.random.randint(10, 40, NUM_PRODUCTS)
# })

# # 3. Generate Fact Tables (Foreign Keys referencing Dimensions)
# print("Generating Fact Tables...")

# # Orders
# order_ids = [generate_id() for _ in range(NUM_ORDERS)]
# start = datetime(2018, 1, 1)
# end = datetime(2018, 8, 31)

# purchase_dates = [random_date(start, end) for _ in range(NUM_ORDERS)]
# approved_dates = [d + timedelta(hours=random.randint(1, 48)) for d in purchase_dates]
# carrier_dates = [d + timedelta(days=random.randint(1, 3)) for d in approved_dates]
# delivered_dates = [d + timedelta(days=random.randint(2, 10)) for d in carrier_dates]

# orders_df = pd.DataFrame({
#     'order_id': order_ids,
#     'customer_id': [random.choice(customer_ids) for _ in range(NUM_ORDERS)],
#     'order_status': ['delivered'] * NUM_ORDERS, # Keeping it to 'delivered' as per your whitelist
#     'order_purchase_timestamp': purchase_dates,
#     'order_approved_at': approved_dates,
#     'order_delivered_carrier_date': carrier_dates,
#     'order_delivered_customer_date': delivered_dates,
#     'order_estimated_delivery_date': [d + timedelta(days=15) for d in purchase_dates]
# })

# # Order Items
# order_items = []
# for order in order_ids:
#     num_items = random.randint(1, 3)
#     for i in range(num_items):
#         order_items.append({
#             'order_id': order,
#             'order_item_id': i + 1,
#             'product_id': random.choice(product_ids),
#             'seller_id': random.choice(seller_ids),
#             'shipping_limit_date': purchase_dates[0] + timedelta(days=5),
#             'price': round(random.uniform(10.0, 500.0), 2),
#             'freight_value': round(random.uniform(5.0, 50.0), 2)
#         })
# items_df = pd.DataFrame(order_items)

# # Order Payments
# payments = []
# for order in order_ids:
#     payments.append({
#         'order_id': order,
#         'payment_sequential': 1,
#         'payment_type': random.choice(['credit_card', 'boleto', 'voucher', 'debit_card']),
#         'payment_installments': random.randint(1, 10),
#         'payment_value': round(random.uniform(20.0, 600.0), 2)
#     })
# payments_df = pd.DataFrame(payments)

# # Order Reviews
# reviews_df = pd.DataFrame({
#     'review_id': [generate_id() for _ in range(NUM_ORDERS)],
#     'order_id': order_ids, # 1-to-1 relationship for simplicity
#     'review_score': np.random.randint(1, 6, NUM_ORDERS),
#     'review_comment_title': '',
#     'review_comment_message': '',
#     'review_creation_date': delivered_dates,
#     'review_answer_timestamp': [d + timedelta(days=random.randint(1, 3)) for d in delivered_dates]
# })

# # Helpdesk Tickets
# helpdesk_df = pd.DataFrame({
#     'ticket_id': [generate_id() for _ in range(NUM_ORDERS // 2)], # 50% of orders get a ticket
#     'email': [random.choice(crm_df['email'].tolist()) for _ in range(NUM_ORDERS // 2)],
#     'issue_type': [random.choice(['late_delivery', 'damaged_item', 'wrong_item', 'refund_request']) for _ in range(NUM_ORDERS // 2)],
#     'satisfaction_rating': np.random.randint(1, 6, NUM_ORDERS // 2)
# })

# # Geolocation
# geo_df = pd.DataFrame({
#     'geolocation_zip_code_prefix': SHARED_ZIP_CODES * 5, # Duplicated to make 100 row,
#     'geolocation_lat': [random.uniform(-33.0, 5.0) for _ in range(100)], # Approx Brazil Lat bounds
#     'geolocation_lng': [random.uniform(-73.0, -34.0) for _ in range(100)], # Approx Brazil Lng bounds
#     'geolocation_city': [random.choice(['sao paulo', 'rio de janeiro', 'curitiba', 'brasilia']) for _ in range(100)],
#     'geolocation_state': [random.choice(states) for _ in range(100)]
# })

# # 4. Save Directly to MinIO Bucket
# print("Uploading files directly to MinIO bucket...")

# # MinIO connection details from your docker-compose.yml
# storage_options = {
#     "key": "admin",
#     "secret": "password",
#     "client_kwargs": {
#         "endpoint_url": "http://minio:9000" # Docker internal network address
#     }
# }

# # The root path in your MinIO bucket
# MINIO_BASE_PATH = "s3://olist-data/landing"

# # Mapping: (Folder Name, File Name, DataFrame)
# files_to_save = [
#     ('crm', 'crm_identities_syn.csv', crm_df),
#     ('customers', 'olist_customers_dataset_syn.csv', customers_df),
#     ('geolocation', 'olist_geolocation_dataset_syn.csv', geo_df),
#     ('helpdesk', 'helpdesk_tickets_syn.csv', helpdesk_df),
#     ('items', 'olist_order_items_dataset_syn.csv', items_df),
#     ('orders', 'olist_orders_dataset_syn.csv', orders_df),
#     ('payments', 'olist_order_payments_dataset_syn.csv', payments_df),
#     ('products', 'olist_products_dataset_syn.csv', products_df),
#     ('reviews', 'olist_order_reviews_dataset_syn.csv', reviews_df),
#     ('sellers', 'olist_sellers_dataset_syn.csv', sellers_df)
# ]

# for folder, filename, df in files_to_save:
#     # Construct the full S3 path: s3://olist-data/landing/folder/filename.csv
#     s3_path = f"{MINIO_BASE_PATH}/{folder}/{filename}"
    
#     # Write directly to MinIO over the network
#     df.to_csv(s3_path, index=False, storage_options=storage_options)
#     print(f"Uploaded: {s3_path} ({len(df)} rows)")

# print("\nDirect upload to MinIO complete! Check your MinIO UI.")

Generating Dimension Tables...
Generating Fact Tables...
Uploading files directly to MinIO bucket...
Uploaded: s3://olist-data/landing/crm/crm_identities_syn.csv (50 rows)
Uploaded: s3://olist-data/landing/customers/olist_customers_dataset_syn.csv (50 rows)
Uploaded: s3://olist-data/landing/geolocation/olist_geolocation_dataset_syn.csv (100 rows)
Uploaded: s3://olist-data/landing/helpdesk/helpdesk_tickets_syn.csv (50 rows)
Uploaded: s3://olist-data/landing/items/olist_order_items_dataset_syn.csv (205 rows)
Uploaded: s3://olist-data/landing/orders/olist_orders_dataset_syn.csv (100 rows)
Uploaded: s3://olist-data/landing/payments/olist_order_payments_dataset_syn.csv (100 rows)
Uploaded: s3://olist-data/landing/products/olist_products_dataset_syn.csv (20 rows)
Uploaded: s3://olist-data/landing/reviews/olist_order_reviews_dataset_syn.csv (100 rows)
Uploaded: s3://olist-data/landing/sellers/olist_sellers_dataset_syn.csv (10 rows)

Direct upload to MinIO complete! Check your MinIO UI.


In [9]:
import os
import uuid
import random
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pyspark.sql.functions import col

# 1. Setup targeted ID
TARGET_UNIQUE_ID = "0000f6ccb0745a6a4b88665a16c9f078"

# Initialize Spark to fetch the customer's real location

print(f"--- Fetching existing data for Customer: {TARGET_UNIQUE_ID} ---")
try:
    df_gold = spark.read.format("delta").load("s3a://olist-data/gold/customer_360")
    
    # Search for this specific customer
    existing_person = df_gold.filter(col("customer_unique_id") == TARGET_UNIQUE_ID).collect()
    
    if len(existing_person) > 0:
        TARGET_CITY = existing_person[0]["customer_city"]
        TARGET_STATE = existing_person[0]["customer_state"]
        print(f"Target Acquired! Location: {TARGET_CITY}, {TARGET_STATE}")
    else:
        print("WARNING: Customer not found in Gold table. Defaulting to Sao Paulo.")
        TARGET_CITY = "sao paulo"
        TARGET_STATE = "SP"

except Exception as e:
    print(f"Error reading Gold table: {e}")
    print("Defaulting to Sao Paulo.")
    TARGET_CITY = "sao paulo"
    TARGET_STATE = "SP"

print("\n--- Generating new transaction data ---")

def generate_id():
    return uuid.uuid4().hex

# Olist Rule: Every new order requires a NEW customer_id linked to the OLD customer_unique_id
new_customer_id = generate_id()
new_order_id = generate_id()
new_product_id = generate_id()
mock_zip_code = random.randint(10000, 99999)

# Set dates to "Right Now" so the new purchase date stands out in your Gold table
now = datetime.now()
purchase_date = now - timedelta(days=2)
delivered_date = now - timedelta(days=1)

# 1. Customers Table (The bridge between the new order and the old person)
customers_df = pd.DataFrame({
    'customer_id': [new_customer_id],
    'customer_unique_id': [TARGET_UNIQUE_ID],
    'customer_zip_code_prefix': [mock_zip_code],
    'customer_city': [TARGET_CITY],
    'customer_state': [TARGET_STATE]
})

# 2. Orders Table
orders_df = pd.DataFrame({
    'order_id': [new_order_id],
    'customer_id': [new_customer_id],
    'order_status': ['delivered'],
    'order_purchase_timestamp': [purchase_date],
    'order_approved_at': [purchase_date + timedelta(hours=1)],
    'order_delivered_carrier_date': [purchase_date + timedelta(days=1)],
    'order_delivered_customer_date': [delivered_date],
    'order_estimated_delivery_date': [purchase_date + timedelta(days=10)]
})

# 3. Order Items Table (They bought 1 expensive item)
items_df = pd.DataFrame({
    'order_id': [new_order_id],
    'order_item_id': [1],
    'product_id': [new_product_id],
    'seller_id': [generate_id()], # Dummy seller ID to satisfy schema
    'shipping_limit_date': [purchase_date + timedelta(days=3)],
    'price': [1500.00], # High value to easily see lifetime value increase!
    'freight_value': [45.50]
})

# 4. Products Table
products_df = pd.DataFrame({
    'product_id': [new_product_id],
    'product_category_name': ['informatica_acessorios'],
    'product_name_lenght': [45],
    'product_description_lenght': [500],
    'product_photos_qty': [3],
    'product_weight_g': [1200],
    'product_length_cm': [25],
    'product_height_cm': [10],
    'product_width_cm': [15]
})

# 5. Geolocation Table (To ensure the zip code joins properly)
geo_df = pd.DataFrame({
    'geolocation_zip_code_prefix': [mock_zip_code],
    'geolocation_lat': [random.uniform(-33.0, 5.0)], 
    'geolocation_lng': [random.uniform(-73.0, -34.0)], 
    'geolocation_city': [TARGET_CITY],
    'geolocation_state': [TARGET_STATE]
})


# ---------------------------------------------------------
# Upload Directly to MinIO
# ---------------------------------------------------------

storage_options = {
    "key": "admin",
    "secret": "password",
    "client_kwargs": {
        "endpoint_url": "http://minio:9000" 
    }
}

MINIO_BASE_PATH = "s3://olist-data/landing"

files_to_save = [
    ('customers', 'olist_customers_targeted.csv', customers_df),
    ('orders', 'olist_orders_targeted.csv', orders_df),
    ('items', 'olist_order_items_targeted.csv', items_df),
    ('products', 'olist_products_targeted.csv', products_df),
    ('geolocation', 'olist_geolocation_targeted.csv', geo_df)
]

for folder, filename, df in files_to_save:
    s3_path = f"{MINIO_BASE_PATH}/{folder}/{filename}"
    df.to_csv(s3_path, index=False, storage_options=storage_options)
    print(f"Uploaded: {s3_path}")

print("DONE")

--- Fetching existing data for Customer: 0000f6ccb0745a6a4b88665a16c9f078 ---
Target Acquired! Location: belem, PA

--- Generating new transaction data ---
Uploaded: s3://olist-data/landing/customers/olist_customers_targeted.csv
Uploaded: s3://olist-data/landing/orders/olist_orders_targeted.csv
Uploaded: s3://olist-data/landing/items/olist_order_items_targeted.csv
Uploaded: s3://olist-data/landing/products/olist_products_targeted.csv
Uploaded: s3://olist-data/landing/geolocation/olist_geolocation_targeted.csv
DONE


In [8]:


# Updated dictionary to match the '_targeted.csv' filenames and core table list
files_to_check = {
    "Customers":   "s3a://olist-data/landing/customers/olist_customers_targeted.csv",
    "Orders":      "s3a://olist-data/landing/orders/olist_orders_targeted.csv",
    "Items":       "s3a://olist-data/landing/items/olist_order_items_targeted.csv",
    "Products":    "s3a://olist-data/landing/products/olist_products_targeted.csv",
    "Geolocation": "s3a://olist-data/landing/geolocation/olist_geolocation_targeted.csv"
}

for table_name, path in files_to_check.items():
    print(f"--- {table_name} Table ---")
    try:
        # Read the CSV directly from the MinIO landing zone
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .option("multiLine", "true") \
            .option("escape", '"') \
            .load(path)
        
        print(f"File Path: {path}")
        print(f"Row Count: {df.count()}")
        
        # Display the data
        df.show(truncate=False)
        print("\n")
        
    except Exception as e:
        print(f"WARNING: Could not load {table_name}. The file may not have uploaded yet.")
        print(f"Error details: {e}\n")

--- Customers Table ---
File Path: s3a://olist-data/landing/customers/olist_customers_targeted.csv
Row Count: 1
+--------------------------------+--------------------------------+------------------------+-------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city|customer_state|
+--------------------------------+--------------------------------+------------------------+-------------+--------------+
|4842149b1d5741aa9b54f41f3c12af87|0000f6ccb0745a6a4b88665a16c9f078|85486                   |belem        |PA            |
+--------------------------------+--------------------------------+------------------------+-------------+--------------+



--- Orders Table ---
File Path: s3a://olist-data/landing/orders/olist_orders_targeted.csv
Row Count: 1
+--------------------------------+--------------------------------+------------+--------------------------+--------------------------+----------------------------+---------